# Comprehensive RAG Validation Pipeline with LangChain + RAGAS

This notebook builds an end-to-end Retrieval-Augmented Generation (RAG) validation workflow using LangChain and RAGAS.


## 1) Install dependencies (run once)

Uncomment and run if packages are missing.

In [ ]:
# %pip install -U ragas langchain langchain-community langchain-openai langchain-huggingface langchain-text-splitters faiss-cpu pypdf pandas datasets tqdm sentence-transformers


## 2) Imports and setup

In [ ]:
import os
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
from datasets import Dataset

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy

# RAGAS-LangChain wrappers (support both old/new ragas import paths)
try:
    from ragas.llms import LangchainLLMWrapper
except Exception:
    from ragas.llms.base import LangchainLLMWrapper

try:
    from ragas.embeddings import LangchainEmbeddingsWrapper
except Exception:
    from ragas.embeddings.base import LangchainEmbeddingsWrapper

pd.set_option("display.max_colwidth", 200)


## 3) Configuration

Choose model providers and paths. For easiest setup, keep `USE_OPENAI = True`.


In [ ]:
USE_OPENAI = True

DATA_DIR = Path("../data")  # put your corpus files here (.txt, .md, .pdf)
ARTIFACTS_DIR = Path("../artifacts/rag_eval")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4

OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

LOCAL_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if USE_OPENAI and not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Set it in your environment before running.")

print("Config loaded")
print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")


## 4) Load corpus documents

Supported: `.txt`, `.md`, `.pdf`.


In [ ]:
def load_documents(data_dir: Path) -> List[Document]:
    docs: List[Document] = []
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir.resolve()}")

    txt_loader = DirectoryLoader(str(data_dir), glob="**/*.txt", loader_cls=TextLoader, show_progress=True)
    md_loader = DirectoryLoader(str(data_dir), glob="**/*.md", loader_cls=UnstructuredMarkdownLoader, show_progress=True)
    pdf_loader = DirectoryLoader(str(data_dir), glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True)

    for loader in [txt_loader, md_loader, pdf_loader]:
        try:
            docs.extend(loader.load())
        except Exception as e:
            print(f"Loader warning: {e}")

    if not docs:
        raise ValueError(f"No documents found in {data_dir.resolve()}. Add .txt/.md/.pdf files.")

    for d in docs:
        d.metadata = d.metadata or {}
        d.metadata["source"] = d.metadata.get("source", "unknown")

    return docs


raw_documents = load_documents(DATA_DIR)
print(f"Loaded {len(raw_documents)} raw documents")
raw_documents[:2]


## 5) Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_documents)
print(f"Created {len(chunks)} chunks")
chunks[:2]


## 6) Embeddings + Vector Index

In [ ]:
if USE_OPENAI:
    embedding_model = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)
else:
    embedding_model = HuggingFaceEmbeddings(model_name=LOCAL_EMBEDDING_MODEL)

vectorstore = FAISS.from_documents(chunks, embedding=embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print("Vectorstore created and retriever ready")


## 7) RAG chain (retrieval + generation)

In [ ]:
if not USE_OPENAI:
    raise ValueError("This notebook currently uses OpenAI for generation. Set USE_OPENAI=True or swap in another LangChain chat model.")

llm = ChatOpenAI(model=OPENAI_CHAT_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    """
You are a precise QA assistant.
Answer the question only from the provided context.
If the answer is not in context, say "I don't know based on the provided context."

Context:
{context}

Question: {question}
Answer:
"""
)

parser = StrOutputParser()


def format_context(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


def rag_answer(question: str) -> Dict[str, Any]:
    retrieved_docs = retriever.invoke(question)
    context_text = format_context(retrieved_docs)
    chain = prompt | llm | parser
    answer = chain.invoke({"context": context_text, "question": question})
    return {
        "question": question,
        "answer": answer,
        "contexts": [d.page_content for d in retrieved_docs],
        "retrieved_docs": retrieved_docs,
    }


## 8) Define evaluation questions and reference answers

Create your golden set. Start with 20-50 high-quality Q/A pairs for stable metrics.


In [ ]:
sample_eval_csv = DATA_DIR / "evaluation_questions_sample.csv"

if sample_eval_csv.exists():
    eval_df = pd.read_csv(sample_eval_csv)
    expected_cols = ["question", "ground_truth"]
    if not set(expected_cols).issubset(set(eval_df.columns)):
        raise ValueError(f"{sample_eval_csv} must contain columns: {expected_cols}")
    evaluation_samples = eval_df[expected_cols].to_dict(orient="records")
    print(f"Loaded {len(evaluation_samples)} evaluation samples from {sample_eval_csv}")
else:
    evaluation_samples = [
        {
            "question": "Replace with your question 1",
            "ground_truth": "Replace with expected answer 1",
        },
        {
            "question": "Replace with your question 2",
            "ground_truth": "Replace with expected answer 2",
        },
    ]
    print("Sample CSV not found; using placeholder evaluation_samples.")

pd.DataFrame(evaluation_samples)


## 9) Run RAG pipeline for each question

In [ ]:
records = []

for sample in evaluation_samples:
    out = rag_answer(sample["question"])
    records.append(
        {
            "question": sample["question"],
            "answer": out["answer"],
            "contexts": out["contexts"],
            "ground_truth": sample["ground_truth"],
        }
    )

pred_df = pd.DataFrame(records)
pred_df.head()


## 10) Build RAGAS dataset + run robust RAGAS validation

In [ ]:
if pred_df.empty:
    raise ValueError("pred_df is empty. Run the RAG inference step first.")

required_cols = {"question", "answer", "contexts", "ground_truth"}
missing_cols = required_cols - set(pred_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns for RAGAS: {missing_cols}")

ragas_dataset = Dataset.from_pandas(pred_df[list(required_cols)])

metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
]

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

result_df = result.to_pandas()
result_df


## 11) Analyze RAGAS scores and failure cases

In [ ]:
display(result_df.describe(include="all"))

ranking_cols = [
    c for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    if c in result_df.columns
]

if ranking_cols:
    summary = result_df[ranking_cols].mean().sort_values(ascending=False)
    print("Average metric scores:")
    display(summary.to_frame("avg_score"))

    result_df["mean_score"] = result_df[ranking_cols].mean(axis=1)
    worst_cases = result_df.sort_values("mean_score", ascending=True).head(5)
    display(worst_cases[["question", "answer", "ground_truth", *ranking_cols, "mean_score"]])
else:
    print("No ranking columns found in result_df.")


## 12) Persist artifacts

In [ ]:
pred_path = ARTIFACTS_DIR / "rag_predictions.csv"
scores_path = ARTIFACTS_DIR / "ragas_scores.csv"

pred_df.to_csv(pred_path, index=False)
result_df.to_csv(scores_path, index=False)

print(f"Saved predictions: {pred_path.resolve()}")
print(f"Saved ragas scores: {scores_path.resolve()}")


## 13) Recommended validation cadence

- Re-run this notebook when you change prompts, chunking, retriever config, embeddings, or model.
- Track scores over time in a versioned artifact folder.
- Add domain-specific custom metrics and human spot checks for high-risk questions.
